# Phase 3: preparing the images

**CRISP-DM stage:** Data preparation

**Runs on:** this first part runs on the laptop's processor, on 2 photos only, because it only demonstrates settings. No model runs. The laptop's PyTorch version differs from Kaggle's, so this is a demonstration, not a reported result. The real prepared batch is checked on Kaggle later.

**Figures with dataset images:** files named `gallery_*` show real parts, so they stay on the laptop and out of git. Each figure is saved and closed, not displayed, so no image ends up in the notebook.

## 1. Find the photos

Both photos are defective, in `regular` lighting, and already in the laptop sample. The code checks that each is in half A, because half B stays sealed.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from anomalib.data.utils import read_image, read_mask
from anomalib.models import Padim, Patchcore, ReverseDistillation, WinClip
from torchvision.tv_tensors import Image as TvImage
from torchvision.tv_tensors import Mask as TvMask

# Lets the notebook run from the project folder or from the notebooks folder
PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent

SAMPLE_FOLDER = PROJECT_FOLDER / "data" / "sample"
RESULTS_FOLDER = PROJECT_FOLDER / "outputs" / "results" / "03_preprocessing"
FIGURES_FOLDER = PROJECT_FOLDER / "outputs" / "figures"
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

PHOTOS = {"sheet_metal": "000_regular.png", "vial": "001_regular.png"}
FIGURE_DPI = 150

half_a_names = []
split_file = PROJECT_FOLDER / "outputs" / "results" / "02_exploration" / "half_a_files.csv"
with open(split_file, newline="") as list_file:
    for row in csv.DictReader(list_file):
        if row["label"] == "bad":
            half_a_names.append((row["product"], row["file_name"]))

for product_name in PHOTOS:
    if (product_name, PHOTOS[product_name]) not in half_a_names:
        raise ValueError("This photo is not in half A: " + PHOTOS[product_name])
print("Both photos are in half A")

## 2. anomalib's default recipes

Each method has a `configure_pre_processor` function, and calling it with no arguments gives its default recipe. This prints the real settings, so nothing is guessed.

- **`Resize`** changes the photo to a fixed size. The size is given as `[height, width]`, so a non-square photo gets squashed. `antialias=True` smooths pixels when shrinking, so fine patterns do not break up.
- **`interpolation`** is how new pixels are worked out from old ones. Bilinear blends the nearest 4 pixels, and bicubic blends the nearest 16.
- **`Normalize`** changes the pixel numbers, explained in part 3.
- **There is no crop** in these default recipes. PatchCore only crops if `center_crop_size` is passed.

In [ ]:
result_lines = []

METHODS = [
    ("PaDiM", Padim),
    ("PatchCore", Patchcore),
    ("Reverse Distillation", ReverseDistillation),
    ("WinCLIP", WinClip),
]

for method_name, method in METHODS:
    result_lines.append("=== " + method_name + " default recipe")
    result_lines.append(str(method.configure_pre_processor().transform))

for line in result_lines:
    print(line)

## 3. Normalization, checked by hand

For each colour channel, normalization does this sum:

`new value = (old value - mean) / std`

The mean is the average brightness of the network's training photos, and std is how much that brightness typically varies. This re-centres a photo's numbers onto the scale the network learned on.

The small test normalizes one made-up pixel with anomalib's own step, then with the sum by hand. The two must match.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TOLERANCE = 0.00001


def normalize_by_hand(value, mean, std):
    """Do the normalization sum for one number."""
    return (value - mean) / std


# The recipe is a list of steps. Position 1 is its Normalize step
imagenet_recipe = Padim.configure_pre_processor().transform
normalize_step = imagenet_recipe.transforms[1]

# A made-up single pixel with three channels, each 0.5
test_pixel = torch.full((1, 3, 1, 1), 0.5)
test_result = normalize_step(test_pixel)

for channel in range(3):
    by_hand = normalize_by_hand(0.5, IMAGENET_MEAN[channel], IMAGENET_STD[channel])
    if abs(float(test_result[0, channel, 0, 0]) - by_hand) > TOLERANCE:
        raise ValueError("anomalib's normalization differs from the sum by hand")
print("Normalization test passed:", test_result.flatten().tolist())

## 4. Before and after, on real photos

`TvImage` and `TvMask` label which is the photo and which is the mask. The resize step uses this label to treat them differently, and the printed mask values below show whether the mask stays pure 0 and 1.

The worked pixel is the centre of the resized photo. All three stored channels are equal, because the photos are greyscale, and each becomes a different number because each channel has its own mean and std.

**The figure:** left is the original photo, middle is after the default resize, and right is after normalization. The right panel shows one channel with its brightness stretched, because its numbers no longer fit a normal display scale. Red outlines show the mask.

In [ ]:
RECIPES = [
    ("ImageNet recipe, PaDiM, PatchCore, Reverse Distillation", Padim.configure_pre_processor().transform),
    ("CLIP recipe, WinCLIP", WinClip.configure_pre_processor().transform),
]
FIGURE_SIZES = {"sheet_metal": (18, 4), "vial": (13, 6)}

for product_name in PHOTOS:
    file_name = PHOTOS[product_name]
    test_folder = SAMPLE_FOLDER / product_name / product_name / "test_public"
    mask_name = Path(file_name).stem + "_mask.png"

    image = read_image(test_folder / "bad" / file_name, as_tensor=True)
    mask = read_mask(test_folder / "ground_truth" / "bad" / mask_name, as_tensor=True)

    result_lines.append("")
    result_lines.append("=== " + product_name + ", " + file_name)
    result_lines.append("as loaded: photo " + str(list(image.shape)) + ", values "
                        + str(round(float(image.min()), 3)) + " to " + str(round(float(image.max()), 3)))
    result_lines.append("as loaded: mask " + str(list(mask.shape)) + ", defect pixels " + str(int(mask.sum())))

    for recipe_name, recipe in RECIPES:
        resize_step = recipe.transforms[0]
        normalize_step = recipe.transforms[1]
        # One extra dimension is added, because these steps expect a batch of photos
        resized_image, resized_mask = resize_step(TvImage(image.unsqueeze(0)), TvMask(mask.unsqueeze(0)))
        normalized_image = normalize_step(resized_image)

        result_lines.append(recipe_name + ":")
        result_lines.append("  after resize: photo " + str(list(resized_image.shape[1:])) + ", values "
                            + str(round(float(resized_image.min()), 3)) + " to " + str(round(float(resized_image.max()), 3)))
        result_lines.append("  after resize: mask values " + str(torch.unique(resized_mask.int()).tolist())
                            + ", defect pixels " + str(int(resized_mask.sum())))
        result_lines.append("  after normalize: values " + str(round(float(normalized_image.min()), 3))
                            + " to " + str(round(float(normalized_image.max()), 3)))

        # The worked pixel: the centre of the resized photo
        center_row = resized_image.shape[2] // 2
        center_column = resized_image.shape[3] // 2
        means = normalize_step.mean
        stds = normalize_step.std
        for channel in range(3):
            old_value = float(resized_image[0, channel, center_row, center_column])
            new_value = float(normalized_image[0, channel, center_row, center_column])
            by_hand = normalize_by_hand(old_value, means[channel], stds[channel])
            result_lines.append("  centre pixel, channel " + str(channel) + ": (" + str(round(old_value, 4)) + " - " + str(round(means[channel], 4))
                                + ") / " + str(round(stds[channel], 4)) + " = " + str(round(by_hand, 4))
                                + ", anomalib gave " + str(round(new_value, 4)))

        # The figure uses the recipe shared by three of the four methods
        if recipe_name.startswith("ImageNet"):
            figure, axes = plt.subplots(1, 3, figsize=FIGURE_SIZES[product_name])

            axes[0].imshow(image[0], cmap="gray", vmin=0, vmax=1)
            axes[0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
            axes[0].set_title("Original, " + str(image.shape[2]) + " x " + str(image.shape[1]), fontsize=10)

            axes[1].imshow(resized_image[0, 0], cmap="gray", vmin=0, vmax=1)
            axes[1].contour(resized_mask[0], levels=[0.5], colors="red", linewidths=0.8)
            axes[1].set_title("After default resize, " + str(resized_image.shape[3]) + " x " + str(resized_image.shape[2]), fontsize=10)

            # matplotlib's default stretches this panel from its own lowest to highest value
            axes[2].imshow(normalized_image[0, 0], cmap="gray")
            axes[2].set_title("After normalization, one channel, stretched", fontsize=10)

            for axis in axes:
                axis.axis("off")
            figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A photo. Red outline: mask")
            figure.savefig(FIGURES_FOLDER / ("gallery_preprocessing_" + product_name + ".png"), dpi=FIGURE_DPI, bbox_inches="tight")
            # Closing without showing keeps dataset images out of the notebook
            plt.close(figure)

# Numbers only, with no image content, so this file can be committed
with open(RESULTS_FOLDER / "preprocessing_defaults.txt", "w") as summary_file:
    for line in result_lines:
        summary_file.write(line + "\n")

for line in result_lines:
    print(line)